# Lab 3 Track B: Retention Offer Generation (demo)

Generates retention offers for at-risk customers with RAG over NorthStar's policy corpus (`offer_rag.py`).

**Pipeline:** policy docs → section chunks → Titan Embeddings v2 → FAISS (in memory) → three retrieval passes → Claude Haiku 4.5 draft → verification pass → 3-sentence offer.

**Before running:**
- AWS credentials with Bedrock access to `amazon.titan-embed-text-v2:0` and `us.anthropic.claude-haiku-4-5-20251001-v1:0` (see `docs/bedrock-access-verification.txt`)
- `pip install -r requirements-lab3.txt`
- Run from the repository root or from `notebooks/`

Nothing here needs the Lab 2 AWS stack. A full run costs about $0.05 in Bedrock calls.

In [1]:
import os, sys
# Make the repo root importable whether the notebook runs from notebooks/ or the root.
root = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
sys.path.insert(0, root)

from offer_rag import OfferRAG, describe_customer

rag = OfferRAG()   # defaults: strategy ABD, k=5, temperature 0, verification on
print(f"Indexed {len(rag.chunks)} policy chunks from northstar-policy-docs/")

Indexed 44 policy chunks from northstar-policy-docs/


## Five customers

The first four are the evaluation harness's RAGAS test cases. The fifth is an extra case: an in-store Gold customer, where shipping benefits matter less.

In [2]:
from evaluation_harness import OFFER_TEST_CASES

customers = [(tc.customer_id, tc.customer_context, tc.question) for tc in OFFER_TEST_CASES]
customers.append((
    "CUST-DEMO-05",
    {"churn_probability": 0.77, "loyalty_tier": "Gold", "days_since_last_purchase": 41,
     "total_lifetime_value": 2650.00, "avg_order_value": 120.45,
     "category_diversity_score": 0.25, "online_to_store_ratio": 0.15,
     "top_categories": ["Home", "Toys"]},
    "Generate a retention offer for this Gold member who shops almost entirely in store "
    "for home goods and toys.",
))

C:\Users\Hayde\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
for customer_id, profile, question in customers:
    out = rag.generate(profile, question)
    print("=" * 100)
    print(f"{customer_id}  |  churn {profile['churn_probability']:.0%}  |  {profile['loyalty_tier']}  |  {out['latency_s']}s")
    print("\nINPUT")
    print(describe_customer(profile))
    print("\nRETRIEVED:", "; ".join(out["sections"]))
    print("\nOFFER")
    print(out["answer"])
    print()

CUST-10000317  |  churn 81%  |  Gold  |  2.67s

INPUT
- Loyalty tier (analytics estimate): Gold
- Churn probability: 81%
- Days since last purchase: 47
- Lifetime value: $3,120.40; average order $186.25
- Buys in 4 of 8 categories
- Channel: shops mostly online, so shipping benefits matter (online ratio 0.72)
- Top categories: Outdoor, Footwear
- Recommended category for this offer: Outdoor and Footwear (the offer recommends a category the customer already buys)

RETRIEVED: POL-LOY-011 4. Tier Benefits; KB-FAQ-022 Loyalty; POL-LOY-011 2. Tiers > 2.3 Tier Protection; POL-RET-004 1. Standard Return Window > 1.1 Extended Window for Platinum Members; POL-LOY-011 2. Tiers; POL-LOY-011 5. Promotional Offers > 5.1 Retention Offers; POL-LOY-011 5. Promotional Offers; POL-RET-004 4. Non-Returnable Merchandise; KB-FAQ-022 Returns; KB-FAQ-022 Guidance for Automated Agents; KB-FAQ-022 Shipping; POL-SHP-007 2. Free Shipping Thresholds

OFFER
Retention offer for this at-risk Gold member who shops mo

CUST-10000904  |  churn 74%  |  Bronze  |  2.63s

INPUT
- Loyalty tier (analytics estimate): Bronze
- Churn probability: 74%
- Days since last purchase: 68
- Lifetime value: $312.80; average order $41.15
- Buys in 1 of 8 categories (a narrow shopper; cross-category discovery is an option)
- Channel: shops mostly in store, so shipping benefits matter little (online ratio 0.20)
- Top categories: Apparel
- Recommended category for this offer: Apparel (the offer recommends a category the customer already buys)

RETRIEVED: POL-LOY-011 4. Tier Benefits; KB-FAQ-022 Loyalty; POL-RET-004 1. Standard Return Window > 1.1 Extended Window for Platinum Members; POL-LOY-011 3. Points; POL-LOY-011 2. Tiers; POL-LOY-011 5. Promotional Offers > 5.1 Retention Offers; POL-LOY-011 5. Promotional Offers; POL-RET-004 4. Non-Returnable Merchandise; KB-FAQ-022 Returns; KB-FAQ-022 Guidance for Automated Agents; KB-FAQ-022 Shipping; POL-SHP-007 2. Free Shipping Thresholds

OFFER
Retention offer for this Bronze m

CUST-10000122  |  churn 66%  |  Platinum  |  3.1s

INPUT
- Loyalty tier (analytics estimate): Platinum
- Churn probability: 66%
- Days since last purchase: 39
- Lifetime value: $8,940.00; average order $447.00
- Buys in 6 of 8 categories
- Channel: shops both online and in store (online ratio 0.55)
- Top categories: Outdoor, Electronics, Home
- Recommended category for this offer: Outdoor and Electronics (the offer recommends a category the customer already buys)

RETRIEVED: POL-LOY-011 4. Tier Benefits; KB-FAQ-022 Loyalty; POL-RET-004 1. Standard Return Window > 1.1 Extended Window for Platinum Members; POL-LOY-011 2. Tiers > 2.3 Tier Protection; POL-LOY-011 2. Tiers; POL-LOY-011 5. Promotional Offers > 5.1 Retention Offers; POL-LOY-011 5. Promotional Offers; POL-RET-004 4. Non-Returnable Merchandise; KB-FAQ-022 Returns; KB-FAQ-022 Guidance for Automated Agents; KB-FAQ-022 Shipping; POL-SHP-007 2. Free Shipping Thresholds

OFFER
Retention offer for this high-value Platinum member with

CUST-10000588  |  churn 70%  |  Silver  |  2.46s

INPUT
- Loyalty tier (analytics estimate): Silver
- Churn probability: 70%
- Days since last purchase: 52
- Lifetime value: $1,985.00; average order $132.30
- Buys in 3 of 8 categories
- Channel: shops mostly online, so shipping benefits matter (online ratio 0.61)
- Top categories: Home, Beauty
- Recommended category for this offer: Home and Beauty (the offer recommends a category the customer already buys)
- Tier caution: lifetime value is within 10% of the Gold threshold ($2,000). Official tier uses trailing 12-month spend, so do not state or imply benefits of any tier other than Silver.

RETRIEVED: POL-LOY-011 4. Tier Benefits; KB-FAQ-022 Loyalty; POL-LOY-011 2. Tiers; POL-RET-004 1. Standard Return Window > 1.1 Extended Window for Platinum Members; POL-LOY-011 3. Points; POL-LOY-011 5. Promotional Offers > 5.1 Retention Offers; POL-LOY-011 5. Promotional Offers; POL-RET-004 4. Non-Returnable Merchandise; KB-FAQ-022 Returns; KB-FAQ-0

CUST-DEMO-05  |  churn 77%  |  Gold  |  2.87s

INPUT
- Loyalty tier (analytics estimate): Gold
- Churn probability: 77%
- Days since last purchase: 41
- Lifetime value: $2,650.00; average order $120.45
- Buys in 2 of 8 categories (a narrow shopper; cross-category discovery is an option)
- Channel: shops mostly in store, so shipping benefits matter little (online ratio 0.15)
- Top categories: Home, Toys
- Recommended category for this offer: Home and Toys (the offer recommends a category the customer already buys)

RETRIEVED: POL-LOY-011 4. Tier Benefits; KB-FAQ-022 Loyalty; POL-LOY-011 2. Tiers > 2.3 Tier Protection; POL-RET-004 1. Standard Return Window > 1.1 Extended Window for Platinum Members; POL-LOY-011 2. Tiers; POL-LOY-011 5. Promotional Offers > 5.1 Retention Offers; POL-LOY-011 5. Promotional Offers; POL-RET-004 4. Non-Returnable Merchandise; KB-FAQ-022 Returns; KB-FAQ-022 Guidance for Automated Agents; KB-FAQ-022 Shipping; POL-SHP-007 2. Free Shipping Thresholds

OFFER
Reten

In [4]:
u = rag.usage
print(f"Generation: {u['calls']} Bedrock calls (draft + verification per offer), "
      f"{u['input_tokens']:,} input / {u['output_tokens']:,} output tokens, ~${rag.cost_usd():.4f}")

Generation: 10 Bedrock calls (draft + verification per offer), 35,956 input / 851 output tokens, ~$0.0402


## Evaluation

RAGAS scoring (faithfulness, answer relevancy, context recall) runs through the provided harness, using a Bedrock judge:

```bash
python evaluation_harness.py --track B
```

Guardrail checks (prompt injection, system-prompt disclosure, out-of-scope requests, tier rules):

```bash
python tests/offer_guardrail_checks.py
```

Results and the design rationale are in `docs/lab3-model-design.md`.